In [ ]:
import math

def matrix_copy(A):
    return [row[:] for row in A]

def matrix_size(A):
    return len(A), len(A[0])

def matrix_multiply(A, B):
    n, m = len(A), len(A[0])
    m2, p = len(B), len(B[0])
    if m != m2:
        raise ValueError("Несовместимые размеры матриц")
    
    result = [[0.0] * p for _ in range(n)]
    for i in range(n):
        for j in range(p):
            for k in range(m):
                result[i][j] += A[i][k] * B[k][j]
    return result

def matrix_transpose(A):
    n, m = len(A), len(A[0])
    return [[A[i][j] for i in range(n)] for j in range(m)]

def matrix_vector_multiply(A, v):
    n = len(A)
    result = [0.0] * n
    for i in range(n):
        for j in range(n):
            result[i] += A[i][j] * v[j]
    return result

def vector_norm(v):
    return math.sqrt(sum(x*x for x in v))

def vector_normalize(v):
    norm = vector_norm(v)
    if norm < 1e-12:
        return v
    return [x / norm for x in v]

def identity_matrix(n):
    return [[1.0 if i == j else 0.0 for j in range(n)] for i in range(n)]

def matrix_subtract_lambda(A, lam):
    n = len(A)
    result = matrix_copy(A)
    for i in range(n):
        result[i][i] -= lam
    return result

In [ ]:
def qr_decomposition(A):
    n = len(A)
    Q = [[0.0] * n for _ in range(n)]
    R = [[0.0] * n for _ in range(n)]
    
    columns = [[A[i][j] for i in range(n)] for j in range(n)]
    
    for j in range(n):
        v = columns[j][:]

        for i in range(j):
            q = [Q[k][i] for k in range(n)]
            R[i][j] = sum(q[k] * v[k] for k in range(n))
            for k in range(n):
                v[k] -= R[i][j] * q[k]
        
        R[j][j] = vector_norm(v)
        
        if R[j][j] > 1e-12:
            for k in range(n):
                Q[k][j] = v[k] / R[j][j]
        else:
            for k in range(n):
                Q[k][j] = 0.0
    
    return Q, R

In [ ]:
def qr_eigenvalues(A, max_iter=1000, eps=1e-10):
    n = len(A)
    A = matrix_copy(A)
    
    for _ in range(max_iter):
        Q, R = qr_decomposition(A)
        A = matrix_multiply(R, Q)

        off_diag = 0.0
        for i in range(1, n):
            for j in range(i):
                off_diag += A[i][j] ** 2
        
        if off_diag < eps:
            break

    eigenvalues = [A[i][i] for i in range(n)]
    return eigenvalues

In [ ]:
def gauss_elimination(A, b):
    n = len(A)
    aug = [A[i][:] + [b[i]] for i in range(n)]
    
    for i in range(n):
        max_row = i
        for k in range(i+1, n):
            if abs(aug[k][i]) > abs(aug[max_row][i]):
                max_row = k
        aug[i], aug[max_row] = aug[max_row], aug[i]
        
        if abs(aug[i][i]) < 1e-12:
            continue
        
        for k in range(i+1, n):
            factor = aug[k][i] / aug[i][i]
            for j in range(i, n+1):
                aug[k][j] -= factor * aug[i][j]

    x = [0.0] * n
    for i in range(n-1, -1, -1):
        if abs(aug[i][i]) < 1e-12:
            x[i] = 1.0 
            continue
        x[i] = aug[i][n]
        for j in range(i+1, n):
            x[i] -= aug[i][j] * x[j]
        x[i] /= aug[i][i]
    
    return x


def find_eigenvector(A, lam):
    n = len(A)
    A_minus = matrix_subtract_lambda(A, lam)

    for i in range(n):
        A_minus[i][i] += 1e-10
    
    b = [0.0] * n
    b[0] = 1e-10
    
    v = gauss_elimination(A_minus, b)

    v = vector_normalize(v)
    return v

In [ ]:
def eigenvalues_eigenvectors_no_lib(A, max_iter=1000, eps=1e-10):
    n = len(A)
    
    eigenvalues = qr_eigenvalues(A, max_iter, eps)
    
    eigenvectors = []
    for lam in eigenvalues:
        v = find_eigenvector(A, lam)
        eigenvectors.append(v)
    
    return eigenvalues, eigenvectors